# snATAC-Express Tutorial
### This notebook demonstrates how to use snATAC-Express to predict gene expression from chromatin accessibility data using machine learning.

## Overview
### snATAC-Express runs in two phases:

1. Phase 1: Initial modeling with all peaks and feature importance ranking
2. Phase 2: Refined modeling using only the most important peaks (top 95%)

## 1. Setup and Imports

In [1]:
# snATAC-Express Tutorial: End-to-End Example Using run_multi_test

import os
import sys
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Add the package to the path
sys.path.append(os.path.abspath('snatac_express'))

# Import the specific functions we need
from snatac_express.scripts.data_preprocessing import (
    load_peak_input, 
    load_gex_input, 
    get_pseudobulk
)

print("✅ Environment ready!")

✅ Environment ready!


## 2. Configuration and Data Paths

In [2]:
# Define configuration with correct project directory
import os

# Set the correct project directory
project_dir = '/home/maggiebrown/projects/snATAC-Express'
print(f"🏠 Project directory: {project_dir}")

config = {
    'input_dir': os.path.join(project_dir, 'example_data', 'input_data'),
    'output_dir': os.path.join(project_dir, 'results', 'tutorial_bach2'),
    'config_yaml': os.path.join(project_dir, 'config.yaml'),
    'gene': 'BACH2'
}

# Create output directory
os.makedirs(config['output_dir'], exist_ok=True)
print(f"📁 Output directory: {config['output_dir']}")

# Check input data
if os.path.exists(config['input_dir']):
    input_files = os.listdir(config['input_dir'])
    print(f"📂 Input files available:")
    for file in input_files:
        print(f"  - {file}")
else:
    print(f"❌ Input directory not found: {config['input_dir']}")

🏠 Project directory: /home/maggiebrown/projects/snATAC-Express
📁 Output directory: /home/maggiebrown/projects/snATAC-Express/results/tutorial_bach2
📂 Input files available:
  - sparse_gex_matrix_colnames.txt
  - group_coverages.csv
  - sparse_gex_matrix_rownames.txt
  - sparse_peak_matrix_colnames.txt
  - sparse_peak_matrix_rownames.txt
  - sparse_gex_matrix.txt.mtx
  - genelist_genebody.txt
  - sparse_peak_matrix.txt.mtx


## 3. Inspect Config File

In [3]:
# View the configuration file. This is the file that contains the parameters for the analysis and may be edited by the user.
print("Configuration file contents:")
print("=" * 50)
with open(config['config_yaml'], 'r') as f:
    print(f.read())

Configuration file contents:
# snATAC-Express Configuration

# General settings
project_name: "snATAC_Express_Analysis"
output_dir: "results"
n_jobs: -1  # Number of parallel jobs (-1 = use all cores)
random_seed: 12345

# Input data paths
input_data:
  sparse_gex_matrix: "sparse_gex_matrix.txt.mtx"
  sparse_peak_matrix: "sparse_peak_matrix.txt.mtx"
  group_coverages: "group_coverages.csv"
  gene_list: "genelist_genebody.txt"
  
# Phase 1 settings (Initial modeling and feature ranking)
phase1:
  # Pseudobulk settings
  pseudobulk:
    replicate: "1"  # Which replicate to use (1 or 2)
    min_cells: 10   # Minimum cells per pseudobulk group
    
  # Peak filtering options: Peaks in at least X% of cells to include.
  peak_filters:
    - name: "all_peaks"
      min_sample_presence: 0.0
    - name: "peaks_10pct"
      min_sample_presence: 0.1
    - name: "peaks_50pct" 
      min_sample_presence: 0.5
  
  # WHICH PEAK FILTER TO USE (set this to 0, 1, or 2)
  # 0 = all_peaks (use all peaks r

## 4. Peak at ATAC-seq Peak Data

In [4]:
# Load ATAC-seq peak matrix
print("🔍 Loading ATAC-seq peak data...")

# Load peak data using the package function
peak_data = load_peak_input('sparse_peak_matrix.txt.mtx', input_dir=config['input_dir'])

print(f"�� Peak matrix shape: {peak_data.shape}")
print(f"📊 Peak data info:")
print(f"  - Number of peaks: {peak_data.shape[0]}")
print(f"  - Number of cells: {peak_data.shape[1]}")
print(f"  - Data types: {peak_data.dtypes.unique()}")
print(f"  - Memory usage: {peak_data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Show sample of peak data
print("\n📋 Sample peak data (first 3 peaks, first 3 cells):")
print(peak_data.iloc[:3, :min(3, peak_data.shape[1])])

# Check if this is test data
if peak_data.shape[1] <= 2:
    print(f"\n⚠️  This appears to be test data with only {peak_data.shape[1]} cell(s)")
    print("   The tutorial will continue but results may be limited")

🔍 Loading ATAC-seq peak data...
�� Peak matrix shape: (247, 76453)
📊 Peak data info:
  - Number of peaks: 247
  - Number of cells: 76453
  - Data types: [dtype('uint8')]
  - Memory usage: 18.03 MB

📋 Sample peak data (first 3 peaks, first 3 cells):
                        Pool_8#GTTAACGGTGCTTTAC-1  Pool_8#CCTTATGTCGCAACAT-1  \
chr6:89827394-89827894                          0                          0   
chr6:89827897-89828397                          0                          0   
chr6:89828900-89829400                          0                          0   

                        Pool_8#GCATATATCAAACTCA-1  
chr6:89827394-89827894                          0  
chr6:89827897-89828397                          0  
chr6:89828900-89829400                          0  


## 5. Peak at Gene Expression Data

In [5]:
print("🧬 Loading gene expression data...")

# Only pass the matrix file name and input_dir
gex_data = load_gex_input('sparse_gex_matrix.txt.mtx', input_dir=config['input_dir'])

print(f"📈 Gene expression matrix shape: {gex_data.shape}")
print(f"📊 Gene expression data info:")
print(f"  - Number of genes: {gex_data.shape[0]}")
print(f"  - Number of cells: {gex_data.shape[1]}")
print(f"  - Data types: {gex_data.dtypes.unique()}")

# Check if BACH2 is in the data
bach2_expression = gex_data.loc[gex_data.index == 'BACH2']
if not bach2_expression.empty:
    print(f"\n🎯 BACH2 expression found!")
    print(f"  - Expression values: {bach2_expression.values.flatten()}")
    print(f"  - Mean expression: {bach2_expression.values.mean():.4f}")
    print(f"  - Std expression: {bach2_expression.values.std():.4f}")
    print(f"  - Number cells with BACH2 transcripts: {(bach2_expression != 0).sum().sum()}")


    # Show sample of GEX data
    print("\n📋 Sample GEX data (BACH2, first 3 cells):")
    print(bach2_expression.iloc[:3, :min(20, bach2_expression.shape[1])])

else:
    print(f"\n⚠️  BACH2 not found in gene expression data")
    print(f"Available genes: {list(gex_data.index)}")

🧬 Loading gene expression data...
📈 Gene expression matrix shape: (1, 76453)
📊 Gene expression data info:
  - Number of genes: 1
  - Number of cells: 76453
  - Data types: [dtype('uint8')]

🎯 BACH2 expression found!
  - Expression values: [ 0  0  0 ...  0 16  0]
  - Mean expression: 10.6663
  - Std expression: 21.1203
  - Number cells with BACH2 transcripts: 34057

📋 Sample GEX data (BACH2, first 3 cells):
       Pool_8#GTTAACGGTGCTTTAC-1  Pool_8#CCTTATGTCGCAACAT-1  \
BACH2                          0                          0   

       Pool_8#GCATATATCAAACTCA-1  Pool_8#CCGTGCTGTAGTTGGC-1  \
BACH2                          0                          0   

       Pool_8#CATAACGGTTATGTGG-1  Pool_8#CTGACCAAGTAAGTCC-1  \
BACH2                          0                          0   

       Pool_8#GGATGGCCAAACCTAT-1  Pool_8#GGAGCAAGTCCTTCTC-1  \
BACH2                          0                          0   

       Pool_8#CTCTGTTCAATTAAGG-1  Pool_8#TTAGGCCCATCATGGC-1  \
BACH2              

## 6. Run the Pipeline

In [6]:
# Import the main workflow runner
import os
import glob
import random

# Set a global random seed for reproducibility
SEED = 12345
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print("🚀 Starting snATAC-Express Two-Phase Pipeline...")
print("=" * 50)

# Change to the project directory to ensure relative paths work
original_cwd = os.getcwd()
project_dir = '/home/maggiebrown/projects/snATAC-Express'
os.chdir(project_dir)

try:
    # Set up command line arguments for BOTH phases
    sys.argv = [
        'run_multi_test.py',
        '--config', 'config.yaml',
        '--phase', 'both',  # Run both Phase 1 and Phase 2
        '--gene', 'BACH2'   # Optional: specific gene
    ]
    
    # Import and run the main function
    from snatac_express.scripts.run_multi_test import main as run_workflow
    
    run_workflow()
    print("✅ Two-phase pipeline completed successfully!")
    
except Exception as e:
    print(f"❌ Error during pipeline execution: {e}")
    raise
finally:
    # Change back to original directory
    os.chdir(original_cwd)

2025-06-16 20:50:21,960 - INFO - Starting snATAC-Express workflow
2025-06-16 20:50:21,960 - INFO - Configuration: config.yaml
2025-06-16 20:50:21,961 - INFO - Phase(s) to run: both
2025-06-16 20:50:21,961 - INFO - 
2025-06-16 20:50:21,962 - INFO - PHASE 1: Initial modeling with feature selection
2025-06-16 20:50:21,962 - INFO - ============================================================
2025-06-16 20:50:21,967 - INFO - Loading ATAC peaks...


🚀 Starting snATAC-Express Two-Phase Pipeline...


2025-06-16 20:50:22,114 - INFO - Loading gene expression...
2025-06-16 20:50:22,233 - INFO - Processing 1 genes...
2025-06-16 20:50:22,234 - INFO - Processing gene BACH2
2025-06-16 20:50:23,122 - INFO -   Total peaks: 247
2025-06-16 20:50:23,123 - INFO -   Filtered peaks (≥10% samples): 132
2025-06-16 20:50:23,124 - INFO -   Running random_forest


Average Score (all peaks): 0.4903450402175389


2025-06-16 20:50:34,171 - INFO -     rf_ranker:
2025-06-16 20:50:34,172 - INFO -       All peaks: R² = 0.4903 (132 peaks)
2025-06-16 20:50:34,172 - INFO -       95% peaks: R² = 0.5287 (94 peaks)


...

Average Score (95% peaks): 0.5397322674551486
Average Score (all peaks): 0.6273309230804444


2025-06-16 21:02:11,975 - INFO -     xgb_ranker: R² = 0.6273 (125 peaks)
2025-06-16 21:02:11,976 - INFO -   Running lightgbm


Average Score (95% peaks): 0.6113690733909607
Average Score (all peaks): 0.5711566447218176


2025-06-16 21:02:21,226 - INFO -     lgbm_ranker: R² = 0.5712 (125 peaks)
2025-06-16 21:02:21,227 - INFO - Step 5: Creating Phase 2 master aggregated peak ranks
2025-06-16 21:02:21,228 - INFO - Creating Phase 2 master aggregated peak ranks file...
2025-06-16 21:02:21,230 - INFO -   BACH2: 125 peaks for Phase 2
2025-06-16 21:02:21,232 - INFO -   Saved Phase 2 master aggregated peak ranks to results/aggregated_results/master_aggregated_peak_ranks.csv
2025-06-16 21:02:21,240 - INFO -   Saved Phase 2 aggregation summary to results/aggregated_results/phase2_aggregation_summary.txt
2025-06-16 21:02:21,241 - INFO - Step 6: Summarizing Phase 2 results
2025-06-16 21:02:21,244 - INFO - Saved Phase 2 summary to results/phase2_cv_summary.txt
2025-06-16 21:02:21,244 - INFO - 
Phase 2 Summary Statistics:
2025-06-16 21:02:21,245 - INFO - Total genes analyzed: 1
2025-06-16 21:02:21,246 - INFO - Average R²: 0.5533
2025-06-16 21:02:21,246 - INFO - Median R²: 0.5712
2025-06-16 21:02:21,248 - INFO - 
Phas

Average Score (95% peaks): 0.5777738710546135
✅ Two-phase pipeline completed successfully!


## 7. List and explore output files

In [ ]:
import os
import glob

# List all files in the results directory
os.chdir('/home/maggiebrown/projects/snATAC-Express')
print("=== Results Directory Structure ===")
for root, dirs, files in os.walk("results"):
    level = root.replace("results", '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for f in files:
        print(f"{subindent}{f}")

## 8. Display final run summary


In [ ]:
import pandas as pd

phase2_summary_path = "results/phase2_cv_summary.txt"
if os.path.exists(phase2_summary_path):
    phase2_summary = pd.read_csv(phase2_summary_path, sep='\t')
    print("=== Phase 2 Cross-Validation Summary ===")
    display(phase2_summary.head(10))  # Show first 10 rows
    print(f"\nTotal genes analyzed: {phase2_summary['Gene'].nunique()}")
    print(f"Methods: {phase2_summary['Method'].unique()}")
else:
    print("Phase 2 summary not found!")

## 9. Visualize Top Aggregated Peak Importances

In [ ]:
# Show top aggregated peaks for a gene (e.g., BACH2)
agg_peaks_path = "results/aggregated_results/BACH2/aggregated_peak_importances_exclLR.csv"
if os.path.exists(agg_peaks_path):
    agg_peaks = pd.read_csv(agg_peaks_path)
    print("=== Top Aggregated Peak Importances (Phase 2, BACH2) ===")
    display(agg_peaks.head(10))
else:
    print("Aggregated peak importances file not found!")

# 10. Final model summary

In [ ]:
agg_summary_path = "results/aggregated_results/phase2_aggregated_summary.csv"
if os.path.exists(agg_summary_path):
    agg_summary = pd.read_csv(agg_summary_path)
    print("=== Phase 2 Aggregated Summary ===")
    display(agg_summary.head(10))
else:
    print("Phase 2 aggregated summary not found!")